In [18]:
import pandas as pd

ratings = pd.read_csv("../data/raw/ratings.csv")

print("Shape:", ratings.shape)
ratings.head()

Shape: (100836, 4)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [19]:
ratings.info()

<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [20]:
print("Unique users:", ratings["userId"].nunique())
print("Unique movies:", ratings["movieId"].nunique())

Unique users: 610
Unique movies: 9724


In [21]:
user_movie_matrix = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

print(user_movie_matrix.shape)

user_movie_matrix.head()

(610, 9724)


movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
movie_user_matrix = user_movie_matrix.T

print(movie_user_matrix.shape)

movie_user_matrix.head()

(9724, 610)


userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,NaN,NaN,4.0,NaN,4.5,NaN,NaN,NaN,...,4.0,NaN,4.0,3.0,4.0,2.5,4.0,2.5,3.0,5.0
2,NaN,NaN,NaN,NaN,NaN,4.0,NaN,4.0,NaN,NaN,...,NaN,4.0,NaN,5.0,3.5,NaN,NaN,2.0,NaN,NaN
3,4.0,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
movie_rating_counts = ratings.groupby("movieId")["rating"].count()

print(movie_rating_counts.describe())

count    9724.000000
mean       10.369807
std        22.401005
min         1.000000
25%         1.000000
50%         3.000000
75%         9.000000
max       329.000000
Name: rating, dtype: float64


In [24]:
popular_movies = movie_rating_counts[
    movie_rating_counts >= 10
]

print("Movies with >= 10 ratings:", len(popular_movies))

Movies with >= 10 ratings: 2269


In [25]:
filtered_ratings = ratings[
    ratings["movieId"].isin(popular_movies.index)
]

print(filtered_ratings.shape)

(81116, 4)


In [26]:
filtered_movie_user_matrix = filtered_ratings.pivot_table(
    index="movieId",
    columns="userId",
    values="rating"
)

print(filtered_movie_user_matrix.shape)

(2269, 610)


In [27]:
from sklearn.metrics.pairwise import cosine_similarity

movie_similarity = cosine_similarity(
    filtered_movie_user_matrix.fillna(0)
)

print(movie_similarity.shape)

(2269, 2269)


In [28]:
movie_indices = pd.Series(
    filtered_movie_user_matrix.index,
    index=filtered_movie_user_matrix.index
)

movie_indices.head()

movieId
1    1
2    2
3    3
5    5
6    6
Name: movieId, dtype: int64

In [29]:
movies = pd.read_csv("../data/raw/movies.csv")

movie_lookup = movies.set_index("movieId")

movie_lookup.head()

,title,genres
movieId,,
1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2,Jumanji (1995),Adventure|Children|Fantasy
3,Grumpier Old Men (1995),Comedy|Romance
4,Waiting to Exhale (1995),Comedy|Drama|Romance
5,Father of the Bride Part II (1995),Comedy


In [30]:
def get_collaborative_recommendations(movie_id, top_n=10):

    idx = filtered_movie_user_matrix.index.get_loc(movie_id)

    sim_scores = list(
        enumerate(movie_similarity[idx])
    )

    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )

    sim_scores = sim_scores[1:top_n+1]

    recommended_movie_ids = [
        filtered_movie_user_matrix.index[i[0]]
        for i in sim_scores
    ]

    recommendations = movie_lookup.loc[
        recommended_movie_ids
    ].copy()

    recommendations["similarity_score"] = [
        round(score, 3)
        for _, score in sim_scores
    ]

    return recommendations[
        ["title", "genres", "similarity_score"]
    ]

In [31]:
get_collaborative_recommendations(1)

,title,genres,similarity_score
movieId,,,
3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.573
480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,0.566
780,Independence Day (a.k.a. ID4) (1996),Action|Adventure|Sci-Fi|Thriller,0.564
260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,0.557
356,Forrest Gump (1994),Comedy|Drama|Romance|War,0.547
364,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX,0.541
1210,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,0.541
648,Mission: Impossible (1996),Action|Adventure|Mystery|Thriller,0.539
1265,Groundhog Day (1993),Comedy|Fantasy|Romance,0.534


In [32]:
get_collaborative_recommendations(3114)

,title,genres,similarity_score
movieId,,,
2355,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,0.620
1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,0.573
4306,Shrek (2001),Adventure|Animation|Children|Comedy|Fantasy|Ro...,0.542
4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,0.532
3175,Galaxy Quest (1999),Adventure|Comedy|Sci-Fi,0.516
1580,Men in Black (a.k.a. MIB) (1997),Action|Comedy|Sci-Fi,0.516
2762,"Sixth Sense, The (1999)",Drama|Horror|Mystery,0.504
1682,"Truman Show, The (1998)",Comedy|Drama|Sci-Fi,0.487
3793,X-Men (2000),Action|Adventure|Sci-Fi,0.487


In [33]:
ratings["rating"].describe()

count    100836.000000
mean          3.501557
std           1.042529
min           0.500000
25%           3.000000
50%           3.500000
75%           4.000000
max           5.000000
Name: rating, dtype: float64